**Create Dataset**

In [20]:
TRAINING_DATA_PATH = "training_data/synth_train_data.npz"

In [21]:
import librosa
import torch
import numpy as np
from ml.datasets import AudioRecordingDataset, generate_dataset

In [22]:
X, Y, HV = generate_dataset()
np.savez(file = TRAINING_DATA_PATH, X = X,Y = Y, hv = HV)

**Load Torch Dataset/Dataloader**

In [32]:
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

data = AudioRecordingDataset(TRAINING_DATA_PATH)
val_mask = (data.hv == 0.25)
val_idx = np.where(val_mask)[0]
train_idx = np.where(~val_mask)[0]


train_X = data.X[train_idx]
data.mean = train_X.mean(axis = 0)
data.std = train_X.std(axis = 0) + 1e-8 #for 0 values

print(f"data_mean: {data.mean.shape}| data_std: {data.std.shape}")

train_data = Subset(data, train_idx)
val_data = Subset(data, val_idx)

print(f"train {len(train_data)} | val {len(val_data)}")

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
val_loader = DataLoader(
    dataset = val_data,
    batch_size = 32,
    shuffle = True
)

# batch size x feature_size (64, 84)
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay=0.005)

data_mean: (84,)| data_std: (84,)
train 1760 | val 440


In [33]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(val_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 4.217 | val 3.761| acc 0.364
Epoch   2 | train 3.356 | val 2.814| acc 0.609
Epoch   3 | train 2.480 | val 2.043| acc 0.818
Epoch   4 | train 1.894 | val 1.547| acc 0.920
Epoch   5 | train 1.504 | val 1.238| acc 0.952
Epoch   6 | train 1.256 | val 1.040| acc 0.973
Epoch   7 | train 1.081 | val 0.897| acc 0.977
Epoch   8 | train 0.954 | val 0.790| acc 0.977
Epoch   9 | train 0.866 | val 0.709| acc 0.984
Epoch  10 | train 0.791 | val 0.652| acc 0.982
Epoch  11 | train 0.724 | val 0.600| acc 0.991
Epoch  12 | train 0.680 | val 0.562| acc 0.993
Epoch  13 | train 0.646 | val 0.527| acc 0.995
Epoch  14 | train 0.602 | val 0.503| acc 0.993
Epoch  15 | train 0.595 | val 0.480| acc 0.995
Epoch  16 | train 0.561 | val 0.459| acc 0.995
Epoch  17 | train 0.539 | val 0.443| acc 0.991
Epoch  18 | train 0.528 | val 0.428| acc 0.995
Epoch  19 | train 0.503 | val 0.416| acc 0.993
Epoch  20 | train 0.499 | val 0.405| acc 0.995
Epoch  21 | train 0.477 | val 0.392| acc 0.998
Epoch  22 | t

In [34]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(data.mean, dtype=torch.float32),
    'std' : torch.tensor(data.std, dtype=torch.float32)
}

torch.save(checkpoint, 'models/linear_synth.pth')

Train On NSynth Dataset; Same linear regression model

In [ ]:
from ml.load_validation import load_data


X,Y,meta=load_data(path="nsynth-valid")


In [43]:
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

gss = GroupShuffleSplit(n_splits=1, test_size= 0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, Y, meta['instrument']))
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis = 0) + 1e-8

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

train_data = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(Y_train, dtype=torch.long),
    )
test_data  = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(Y_test, dtype=torch.long),
    )


train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
test_loader = DataLoader(
    dataset = test_data,
    batch_size = 32,
    shuffle = True
)

model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr = 0.001, weight_decay=0.01)

In [44]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in test_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()


            #add overfitting check

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(test_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 2.973 | val 1.566| acc 0.744
Epoch   2 | train 1.300 | val 0.833| acc 0.883
Epoch   3 | train 0.845 | val 0.591| acc 0.900
Epoch   4 | train 0.648 | val 0.482| acc 0.915
Epoch   5 | train 0.542 | val 0.389| acc 0.935
Epoch   6 | train 0.458 | val 0.350| acc 0.930
Epoch   7 | train 0.416 | val 0.322| acc 0.934
Epoch   8 | train 0.372 | val 0.311| acc 0.932
Epoch   9 | train 0.348 | val 0.299| acc 0.932
Epoch  10 | train 0.310 | val 0.292| acc 0.926
Epoch  11 | train 0.295 | val 0.274| acc 0.933
Epoch  12 | train 0.281 | val 0.259| acc 0.931
Epoch  13 | train 0.265 | val 0.244| acc 0.931
Epoch  14 | train 0.253 | val 0.264| acc 0.930
Epoch  15 | train 0.239 | val 0.263| acc 0.926
Epoch  16 | train 0.226 | val 0.266| acc 0.923
Epoch  17 | train 0.217 | val 0.264| acc 0.917
Epoch  18 | train 0.210 | val 0.237| acc 0.926
Epoch  19 | train 0.212 | val 0.254| acc 0.920
Epoch  20 | train 0.195 | val 0.255| acc 0.919
Epoch  21 | train 0.185 | val 0.255| acc 0.926
Epoch  22 | t

In [45]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(train_mean, dtype=torch.float32),
    'std' : torch.tensor(train_std, dtype=torch.float32)
}

torch.save(checkpoint, 'models/linear_nsynth.pth')

**Test Validation**

Load Model

In [46]:
VALIDATION_DATA_PATH = "nsynth-test"
VALIDATION_PROCESSED_DATA = "validation_data"


In [48]:
#Generate Validate dataset
from ml.load_validation import load_data

A,B, _ = load_data()

np.savez(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}", X=A, Y=B)


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=160
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=80
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=152
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=76
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=248
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/lib

  500/4096 scanned, 493 kept


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=200
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=100
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=216
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=108
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=132
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=64
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/li

  1000/4096 scanned, 988 kept


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=192
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=96
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=48
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=24
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=36
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=148
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/libro

...

  2000/4096 scanned, 1984 kept


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=168
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=68
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=60
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=28
  warnings.warn(


  2500/4096 scanned, 2478 kept
  3000/4096 scanned, 2972 kept


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=84
  warnings.warn(


  3500/4096 scanned, 3468 kept
  4000/4096 scanned, 3966 kept
done: 4060 samples from nsynth-test


In [50]:
import numpy as np

# Validation Data on Linear Classification Trained on NSynth Data

checkpoint = torch.load('models/linear_nsynth.pth')

# Define model Architecture - hidden size must match the trained checkpoint
hidden = checkpoint['model_state']['0.weight'].shape[0]
model = nn.Sequential(
    nn.Linear(84, hidden), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(hidden, 88) # 88 = num of valid midi_notes
)

d =  np.load(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}.npz")
#d =  np.load(f"training_data/synth_train_data.npz")
X = torch.tensor(d['X'], dtype  = torch.float32)
Y = torch.tensor(d['Y'], dtype = torch.long)

#norm
X = (X - checkpoint['mean']) / checkpoint['std']

model.load_state_dict(checkpoint['model_state'])
model.eval()

with torch.no_grad():
    logits = model(X)

acc = (logits.argmax(1) == Y).float().mean().item()

print(f"val acc {acc:.3f}")

val acc 0.952
